# 05 - Rincian Regularisasi
Materi Modul 5: hubungan antara **fungsi kerugian (loss function)** dan berbagai jenis regularisasi.

Fungsi kerugian yang diminimalkan:

- OLS   : $\sum (y_i - \hat{y}_i)^2$
- Ridge : $\sum (y_i - \hat{y}_i)^2 + \alpha \sum \beta_j^2$
- Lasso : $\sum (y_i - \hat{y}_i)^2 + \alpha \sum |\beta_j|$
- Elastic Net : $\sum (y_i - \hat{y}_i)^2 + \alpha \left[ \rho \sum |\beta_j| + (1-\rho) \sum \beta_j^2 \right]$ , dengan $\rho$ = `l1_ratio`

Catatan: sklearn memakai skala berbeda untuk tiap model (misal Lasso membagi dengan 2n), jadi di notebook ini rumus dihitung manual hanya untuk Ridge supaya hasilnya persis sama.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression, Ridge, Lasso, ElasticNet, LassoCV
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error

df = pd.read_csv('../datasets/diabetes.csv')
X = df.drop(columns='progres_penyakit')
y = df['progres_penyakit']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

scaler = StandardScaler().fit(X_train)
Xs_train = scaler.transform(X_train)
Xs_test = scaler.transform(X_test)

## 1. Hitung fungsi kerugian secara manual

In [ ]:
def loss_ridge(beta, b0, X, y, alpha):
    pred = X @ beta + b0
    return np.sum((y - pred) ** 2) + alpha * np.sum(beta ** 2)

def loss_lasso(beta, b0, X, y, alpha):
    pred = X @ beta + b0
    return np.sum((y - pred) ** 2) + alpha * np.sum(np.abs(beta))

ols = LinearRegression().fit(Xs_train, y_train)
ridge = Ridge(alpha=10).fit(Xs_train, y_train)

print('Loss Ridge pada koefisien OLS  :', round(loss_ridge(ols.coef_, ols.intercept_, Xs_train, y_train.values, 10), 1))
print('Loss Ridge pada koefisien Ridge:', round(loss_ridge(ridge.coef_, ridge.intercept_, Xs_train, y_train.values, 10), 1))
print('-> koefisien Ridge memberi loss lebih kecil untuk fungsi kerugian Ridge')

## 2. Solusi tertutup Ridge: $\beta = (X^T X + \alpha I)^{-1} X^T y$
Data di-center dulu supaya intercept terpisah.

In [ ]:
alpha = 10
Xc = Xs_train - Xs_train.mean(axis=0)
yc = y_train.values - y_train.values.mean()

beta_manual = np.linalg.inv(Xc.T @ Xc + alpha * np.eye(Xc.shape[1])) @ Xc.T @ yc
print('Manual :', beta_manual.round(3))
print('sklearn:', ridge.coef_.round(3))
print('Sama persis?', np.allclose(beta_manual, ridge.coef_))

## 3. Efek alpha: dari OLS sampai semua koefisien -> 0

In [ ]:
for a in [0, 0.1, 10, 100, 1000, 100000]:
    if a == 0:
        m = LinearRegression().fit(Xs_train, y_train)
    else:
        m = Ridge(alpha=a).fit(Xs_train, y_train)
    print(f'alpha={a:>7} | sum |koef|={np.abs(m.coef_).sum():8.2f} | max |koef|={np.abs(m.coef_).max():7.2f}')

## 4. Kenapa harus di-scale?
Tanpa scaling, fitur berskala besar punya koefisien kecil -> penaltinya ringan; fitur berskala kecil dikenai penalti berat. Penalti jadi tidak adil.

In [ ]:
sd = X_train.std()
print('Standar deviasi fitur (sebelum scaling):')
print(sd.round(3).to_string())

In [ ]:
ridge_tanpa = Ridge(alpha=10).fit(X_train, y_train)
ridge_dengan = Ridge(alpha=10).fit(Xs_train, y_train)

perbandingan = pd.DataFrame({
    'koef tanpa scaling': ridge_tanpa.coef_,
    'koef dengan scaling': ridge_dengan.coef_,
}, index=X.columns).round(3)
perbandingan

In [ ]:
rmse_tanpa = np.sqrt(mean_squared_error(y_test, ridge_tanpa.predict(X_test)))
rmse_dengan = np.sqrt(mean_squared_error(y_test, ridge_dengan.predict(Xs_test)))
print('RMSE test tanpa scaling :', round(rmse_tanpa, 2))
print('RMSE test dengan scaling:', round(rmse_dengan, 2))

## 5. Ridge vs Lasso: siapa yang membuat koefisien tepat 0?

In [ ]:
alphas = [0.1, 0.5, 1, 2, 5, 10]
for a in alphas:
    jumlah_nol_lasso = np.sum(Lasso(alpha=a, max_iter=10000).fit(Xs_train, y_train).coef_ == 0)
    jumlah_nol_ridge = np.sum(Ridge(alpha=a).fit(Xs_train, y_train).coef_ == 0)
    print(f'alpha={a:>4} | koef = 0 pada Lasso: {jumlah_nol_lasso:2d} | pada Ridge: {jumlah_nol_ridge:2d}')

## 6. Elastic Net: efek `l1_ratio`
`l1_ratio=0` -> seperti Ridge, `l1_ratio=1` -> seperti Lasso.

In [ ]:
for rasio in [0.1, 0.3, 0.5, 0.7, 0.9, 1.0]:
    m = ElasticNet(alpha=1, l1_ratio=rasio, max_iter=10000).fit(Xs_train, y_train)
    nol = np.sum(m.coef_ == 0)
    rmse = np.sqrt(mean_squared_error(y_test, m.predict(Xs_test)))
    print(f'l1_ratio={rasio:.1f} | koef = 0: {nol} | RMSE test={rmse:.2f}')

## 7. Lasso sebagai alat seleksi fitur (alpha dipilih CV)

In [ ]:
lasso_cv = LassoCV(cv=5, random_state=0, max_iter=10000).fit(Xs_train, y_train)
koef = pd.Series(lasso_cv.coef_, index=X.columns).round(2)
print('Alpha terbaik:', round(lasso_cv.alpha_, 4))
koef.sort_values(key=abs, ascending=False)

In [ ]:
koef.sort_values().plot(kind='barh', color=['tomato' if k < 0 else 'steelblue' for k in koef.sort_values()])
plt.title('Koefisien Lasso (alpha terbaik dari CV)')
plt.xlabel('Koefisien (data ter-scale)')
plt.show()

## Ringkasan
| | Ridge | Lasso | Elastic Net |
|---|---|---|---|
| Penalti | L2 (kuadrat) | L1 (absolut) | L1 + L2 |
| Koefisien jadi 0? | Tidak | Ya | Ya |
| Cocok jika | banyak fitur berkorelasi | ingin seleksi fitur | kombinasi keduanya |
| Perlu scaling? | Ya | Ya | Ya |

## Catatan
- (arti alpha besar/kecil, hubungan loss function dengan regularisasi, kapan pilih tiap jenis, dll.)